# SECURECOM Cryptography Demo
**Developer:** Abdul Samad  
**University:** Aror University of Art, Architecture, Design & Heritage, Sukkur  
**Purpose:** Demonstrate authenticated encryption in a local educational simulator.

> Educational only. This notebook does not create a real military communication channel.

## 1. Import the required library
Run this cell first. If `cryptography` is missing, install it using `pip install cryptography`.

In [ ]:
import base64, json, secrets
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag
print('SECURECOM cryptography demo ready.')

## 2. Generate a temporary AES-256 key
The notebook keeps the key in memory for this session. Do not share private keys with encrypted packets.

In [ ]:
key = AESGCM.generate_key(bit_length=256)
print('Key generated. Length:', len(key), 'bytes')
print('Key fingerprint:', __import__('hashlib').sha256(key).hexdigest()[:16].upper())

## 3. Encrypt a message
AES-GCM authenticates ciphertext and the sender/receiver metadata.

In [ ]:
def encrypt_demo(message, key, sender='ALPHA', receiver='BRAVO'):
    nonce = secrets.token_bytes(12)
    aad = f'SECURECOM|1|{sender}|{receiver}'.encode()
    ciphertext = AESGCM(key).encrypt(nonce, message.encode('utf-8'), aad)
    packet = {
        'app': 'SECURECOM', 'version': 1, 'algorithm': 'AES-256-GCM',
        'sender': sender, 'receiver': receiver,
        'nonce_b64': base64.b64encode(nonce).decode(),
        'ciphertext_b64': base64.b64encode(ciphertext).decode()
    }
    return json.dumps(packet, indent=2)

packet = encrypt_demo('Secure test message from ALPHA to BRAVO.', key)
print(packet)

## 4. Authenticate and decrypt

In [ ]:
def decrypt_demo(packet_text, key):
    packet = json.loads(packet_text)
    if packet.get('app') != 'SECURECOM' or packet.get('version') != 1:
        raise ValueError('Unsupported packet')
    nonce = base64.b64decode(packet['nonce_b64'], validate=True)
    ciphertext = base64.b64decode(packet['ciphertext_b64'], validate=True)
    aad = f"SECURECOM|1|{packet['sender']}|{packet['receiver']}".encode()
    return AESGCM(key).decrypt(nonce, ciphertext, aad).decode('utf-8')

print('Decrypted:', decrypt_demo(packet, key))

## 5. Tamper detection test
The modified packet should fail authentication. This demonstrates integrity protection.

In [ ]:
tampered = json.loads(packet)
raw = bytearray(base64.b64decode(tampered['ciphertext_b64']))
raw[0] ^= 1
tampered['ciphertext_b64'] = base64.b64encode(raw).decode()
try:
    decrypt_demo(json.dumps(tampered), key)
except (InvalidTag, ValueError, KeyError) as exc:
    print('PASS: modified packet rejected:', type(exc).__name__)

## 6. Limitations
This demonstration uses an in-memory key and no network transport. It is not a complete secure messaging product; key distribution, user identity, persistent storage, and production threat modeling are outside this prototype.